In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita visualizacao grafica inline no Jupyter notebook
%matplotlib inline

# Inspecionar uma resposta oddball auditiva em EEG real

Utilize o EEGDashDataset para carregar uma execucao de P300 auditivo do OpenNeuro ``ds003061``, inspecione seus rotulos reais e plote os ERPs das condicoes padrao e oddball. O sujeito 001, execucao 2, contem cerca de 63.4 MB de dados de sinal. A selecao explicita da execucao evita baixar as outras duas gravacoes deste participante. Computacao em CPU e suficiente; defina ``EEGDASH_CACHE_DIR`` para reter os downloads entre sessoes.

O objetivo e rastrear um marcador de estimulo observado ate uma resposta mensurada: carregar a gravacao, selecionar os dois tipos de eventos documentados, criar ensaios alinhados e calcular a media de cada condicao. Um ERP e um curso temporal medio de voltagem; ele nao atribui um rotulo previsto a um novo ensaio. Esta pagina termina, portanto, com uma comparacao descritiva e nao com um classificador ajustado.



## 1. Carregar exatamente uma gravacao e inspecionar suas anotacoes
Identificadores de sujeito sao strings: preserve os zeros a esquerda em ``001``. Os filtros de tarefa e execucao delimitam a aquisicao antes que qualquer amostra seja lida. Imprimir as contagens de anotacoes torna o vocabulario da fonte visivel e evita tratar respostas motoras ou marcadores nao relacionados como estimulos padrao acidentalmente.



In [ ]:
# Importa utilitarios de sistema operacional e caminhos
import os
from pathlib import Path

# Importa bibliotecas para plotagem, MNE para dados eletrofisiologicos, arrays numericos e DataFrames
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
# Importa transformacoes de referencia media e remocao de offset DC do Braindecode
from braindecode.preprocessing import RemoveCommonAverageReference, RemoveDCOffset

# Importa dataset do EEGDash e funcao para media do sinal
from eegdash import EEGDashDataset
from eegdash.features import signal_mean

# Define diretorio de cache
cache_dir = Path(os.environ.get("EEGDASH_CACHE_DIR", ".eegdash_cache"))
# Instancia o dataset carregando apenas a gravacao desejada do sujeito 001
dataset = EEGDashDataset(
    cache_dir=cache_dir,
    dataset="ds003061",
    subject="001",
    task="P300",
    run="2",
    n_jobs=1,
)
assert len(dataset.datasets) == 1
# Carrega o objeto Raw em memoria RAM e seleciona os canais EEG
raw = dataset.datasets[0].raw.copy().load_data().pick("eeg")
print(pd.Series(raw.annotations.description).value_counts())
# Preserva a grafia literal dos marcadores da fonte, incluindo eventuais erros de digitacao originais
mapping = {"stimulus/standard": 1, "stimulus/oddball_with_reponse": 2}
assert set(mapping) <= set(raw.annotations.description)
assert "Cz" in raw.ch_names, "This ERP comparison requires Cz"

## 2. Filtrar e segmentar epocas em relacao ao inicio real do estimulo
Cada epoca abrange de 100 ms antes a 600 ms apos o estimulo, com subtracao da media pre-estimulo (-100 a 0 ms). O sinal e filtrado de 0.5 a 30 Hz e reamostrado a 128 Hz apos a remocao de offset DC e aplicacao da referencia media comum.



In [ ]:
# Preserva anotacoes e metadados de medicao originais
source_annotations = raw.annotations.copy()
source_date = raw.info["meas_date"]
source_grid = (raw.info["sfreq"], raw.n_times, raw.first_samp)
# Remove offset DC e aplica referencia media comum
RemoveDCOffset().apply(raw)
RemoveCommonAverageReference().apply(raw)
assert (raw.info["sfreq"], raw.n_times, raw.first_samp) == source_grid
raw.set_meas_date(source_date)
raw.set_annotations(source_annotations)
# Aplica filtro passa-banda de 0.5 a 30 Hz
raw.filter(0.5, 30.0)
# Extrai eventos e cria as epocas alinhadas
events, _ = mne.events_from_annotations(raw, event_id=mapping)
epochs = mne.Epochs(
    raw,
    events,
    event_id={"standard": 1, "oddball": 2},
    tmin=-0.1,
    tmax=0.6,
    baseline=(-0.1, 0),
    preload=True,
    reject_by_annotation=True,
)
# Reamostra para 128 Hz
epochs.resample(128)
assert np.isfinite(epochs.get_data()).all()
assert all(len(epochs[name]) > 1 for name in epochs.event_id)
print({name: len(epochs[name]) for name in epochs.event_id})
print("Epoch shape:", epochs.get_data().shape)

## 3. Plotar a resposta medida no eletrodo Cz
A media das epocas e calculada separadamente para cada condicao e exibida no canal central Cz, onde o componente P300 e a diferenca oddball-minus-standard sao avaliados entre 250 e 400 ms.



In [ ]:
# Calcula a resposta media (ERP) para cada condicao
evokeds = {name: epochs[name].average() for name in epochs.event_id}
# Plota a comparacao das formas de onda no eletrodo Cz
mne.viz.plot_compare_evokeds(evokeds, picks="Cz", show=False)
# Calcula a diferenca de potenciais entre a condicao oddball e a padrao
difference = mne.combine_evoked([evokeds["oddball"], evokeds["standard"]], [1, -1])
interval = (difference.times >= 0.25) & (difference.times <= 0.4)
cz = difference.ch_names.index("Cz")
# Imprime a amplitude media da diferenca na janela de 250 a 400 ms em microvolts
print(
    "Mean oddball-minus-standard at Cz, 250–400 ms (µV):",
    signal_mean(difference.data[cz, interval]) * 1e6,
)
plt.show()

## Conclusao
Inspecionamos e tracamos a curva temporal do potencial evocado auditivo oddball (P300) a partir de estimulos reais, calculando diferencas de voltagem de forma descritiva e reprodutivel.

